# HITL Pipeline — ML Classifier Training

This notebook trains one binary classifier per pipeline stage using **Mode B human-review data**.

| Stage | Classifier | Predicts |
|-------|-----------|----------|
| Script | `script_classifier.pkl` | Will a human reviewer **reject** the script? |
| Audio  | `audio_classifier.pkl`  | Will a human reviewer **reject** the audio? |
| Visual | `visual_classifier.pkl` | Will a human reviewer **reject** the visual? |
| Video  | `video_classifier.pkl`  | Will a human reviewer **reject** the final video? |

**Label encoding:** `reject = 1`, `accept = 0`

**Mode C** loads these models to predict rejection probability at each stage.  
Human review is triggered only when predicted probability ≥ 0.65 (`ML_REJECTION_THRESHOLD`).

In [ ]:
# Install required packages
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'scikit-learn', 'xgboost', 'joblib',
    'imbalanced-learn', 'pandas', 'numpy',
    'matplotlib', 'seaborn', '--quiet'
])

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('WARNING: XGBoost not available — only LR and RF will be trained.')

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Imports OK')

In [ ]:
CSV_PATH = r'D:\dessertation\shorts_automation\mode_b_export.csv'

df_raw = pd.read_csv(CSV_PATH)

print(f'Shape: {df_raw.shape}')
print(f'\nColumns ({len(df_raw.columns)}):')
print(df_raw.columns.tolist())
print('\n--- First 5 rows ---')
display(df_raw.head())
print('\nhuman_decision value counts:')
print(df_raw['human_decision'].value_counts())
print('\nstage_name value counts:')
print(df_raw['stage_name'].value_counts())

## Data Overview

- **582 rows** — one row per stage attempt with a recorded human decision
- **Class imbalance:** accepts (442) outnumber rejects (140) by ~3:1
- All four pipeline stages are represented
- Features are stage-specific: script rows have NaN in audio/visual/video columns and vice versa
- `class_weight='balanced'` and `scale_pos_weight` are used to compensate for imbalance

In [ ]:
# ── Prepare working dataframe ──────────────────────────────────────────────
df = df_raw[df_raw['mode'] == 'B'].copy()
df = df[df['human_decision'].notna()].copy()

# Label encoding: reject=1, accept=0
df['label'] = (df['human_decision'] == 'reject').astype(int)

# ── Plot 1: Accept vs Reject per stage ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

stage_decision = (
    df.groupby(['stage_name', 'human_decision'])
    .size()
    .unstack(fill_value=0)
)
stage_decision.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#F44336'],
                    edgecolor='white', width=0.6)
axes[0].set_title('Accept vs Reject counts per stage', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Stage')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Decision')
for bar in axes[0].patches:
    h = bar.get_height()
    if h > 0:
        axes[0].text(bar.get_x() + bar.get_width() / 2, h + 1,
                     str(int(h)), ha='center', va='bottom', fontsize=9)

# ── Plot 2: Missing values heatmap ────────────────────────────────────────
feature_cols_all = [
    'readability_score','lexical_diversity','prompt_coverage','sentence_redundancy',
    'entity_consistency','topic_coherence','factual_conflict_flag','prompt_ambiguity',
    'phoneme_error_rate','silence_ratio','speaking_rate_variance','energy_variance',
    'tts_word_count_match','clip_similarity','aesthetic_score','blur_score',
    'object_match_score','colour_tone_match','av_sync_error_ms','transition_smoothness',
    'duration_deviation_s','prior_stage_corrections','cumulative_risk_score','api_retry_count',
]
missing_matrix = pd.DataFrame(
    {stage: df[df['stage_name'] == stage][feature_cols_all].isna().mean()
     for stage in ['script', 'audio', 'visual', 'video']}
)
sns.heatmap(missing_matrix, ax=axes[1], cmap='YlOrRd', vmin=0, vmax=1,
            annot=True, fmt='.1%', linewidths=0.4, annot_kws={'size': 7})
axes[1].set_title('Missing value rate per feature per stage', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

## Feature Definitions

Each stage uses only its own quality metrics plus three **cross-stage** features shared by all stages:

| Cross-stage feature | Meaning |
|--------------------|---------|
| `prior_stage_corrections` | Number of human rejections in earlier stages of this run |
| `cumulative_risk_score` | Running risk score propagated from earlier stages |
| `api_retry_count` | Number of API retries in this stage |

Null values are filled with the **column mean** before training so no rows are dropped.

In [ ]:
CROSS_STAGE = ['prior_stage_corrections', 'cumulative_risk_score', 'api_retry_count']

script_features = [
    'readability_score', 'lexical_diversity', 'prompt_coverage',
    'sentence_redundancy', 'entity_consistency', 'topic_coherence',
    'factual_conflict_flag', 'prompt_ambiguity',
] + CROSS_STAGE

audio_features = [
    'phoneme_error_rate', 'silence_ratio', 'speaking_rate_variance',
    'energy_variance', 'tts_word_count_match',
] + CROSS_STAGE

visual_features = [
    'clip_similarity', 'aesthetic_score', 'blur_score',
    'object_match_score', 'colour_tone_match',
] + CROSS_STAGE

video_features = [
    'av_sync_error_ms', 'transition_smoothness', 'duration_deviation_s',
] + CROSS_STAGE

STAGE_FEATURE_MAP = {
    'script': script_features,
    'audio':  audio_features,
    'visual': visual_features,
    'video':  video_features,
}

print('Feature sets:')
for stage, feats in STAGE_FEATURE_MAP.items():
    n = len(df[df['stage_name'] == stage])
    r = int(df[df['stage_name'] == stage]['label'].sum())
    a = n - r
    print(f'  {stage:<8} {n:>4} rows  accept={a}  reject={r}  features({len(feats)}): {feats}')

## Training Setup

For each stage we train **three classifiers** and select the best by F1 score:

1. `LogisticRegression` — fast linear baseline with `class_weight='balanced'`
2. `RandomForestClassifier` — ensemble tree model with `class_weight='balanced'`
3. `XGBClassifier` — gradient boosting with `scale_pos_weight` to up-weight rejects

**Why F1?** Accuracy is misleading with imbalanced classes (~75% accept baseline).  
F1 penalises both missed rejects (false negatives) and false alarms (false positives).

In [ ]:
def train_stage_classifiers(stage_name, feature_cols, df):
    """
    Train LR, RF, and XGBoost for one pipeline stage.
    Returns (best_model, best_model_name, all_metrics, valid_cols, X_test, y_test).
    """
    stage_df = df[df['stage_name'] == stage_name].copy()

    # Keep only columns present and with at least one non-null value
    valid_cols = [c for c in feature_cols
                  if c in stage_df.columns and stage_df[c].notna().sum() > 0]

    X = stage_df[valid_cols].fillna(stage_df[valid_cols].mean())
    y = stage_df['label'].values

    n_reject = int((y == 1).sum())
    n_accept = int((y == 0).sum())
    spw = n_accept / max(n_reject, 1)  # scale_pos_weight for XGBoost

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if n_reject > 1 else None
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    models = {
        'LogisticRegression': LogisticRegression(
            class_weight='balanced', max_iter=1000, random_state=42),
        'RandomForest': RandomForestClassifier(
            n_estimators=100, class_weight='balanced', random_state=42),
    }
    if XGBOOST_AVAILABLE:
        models['XGBoost'] = XGBClassifier(
            scale_pos_weight=spw, random_state=42,
            eval_metric='logloss', n_estimators=100)

    all_metrics = {}
    best_f1, best_name, best_model = -1.0, '', None

    for name, model in models.items():
        if name == 'LogisticRegression':
            model.fit(X_train_sc, y_train)
            y_pred = model.predict(X_test_sc)
            y_prob = model.predict_proba(X_test_sc)[:, 1]
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            'accuracy':  round(float(accuracy_score(y_test, y_pred)), 4),
            'precision': round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
            'recall':    round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
            'f1':        round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
            'roc_auc':   round(float(roc_auc_score(y_test, y_prob)) if y_test.sum() > 0 else 0.5, 4),
            'cm':        confusion_matrix(y_test, y_pred),
            'model_obj': model,
            'feature_importances': getattr(model, 'feature_importances_', None),
        }
        all_metrics[name] = metrics

        if metrics['f1'] > best_f1:
            best_f1, best_name, best_model = metrics['f1'], name, model

    # ── Summary table ─────────────────────────────────────────────────────
    print(f"\n{'='*62}")
    print(f"  STAGE: {stage_name.upper()}   rows={len(stage_df)}"
          f"   reject(1)={n_reject}   accept(0)={n_accept}")
    print(f"  XGBoost scale_pos_weight = {spw:.3f}")
    print(f"{'='*62}")
    hdr = f"  {'Model':<22} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}"
    print(hdr)
    print(f"  {'-'*58}")
    for mname, m in all_metrics.items():
        tag = ' <-- BEST' if mname == best_name else ''
        print(f"  {mname:<22} {m['accuracy']:>6.3f} {m['precision']:>6.3f}"
              f" {m['recall']:>6.3f} {m['f1']:>6.3f} {m['roc_auc']:>6.3f}{tag}")
    print(f"{'='*62}")

    return best_model, best_name, all_metrics, valid_cols, X_test, y_test


def plot_confusion_matrix(cm, stage_name, model_name, ax=None):
    """Plot a labelled confusion matrix heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred accept(0)', 'Pred reject(1)'],
                yticklabels=['Act accept(0)', 'Act reject(1)'],
                linewidths=0.5, cbar=False)
    ax.set_title(f'{stage_name} — {model_name}', fontsize=10, fontweight='bold')


def plot_top_importances(metrics_dict, valid_cols, stage_name, top_n=5):
    """Side-by-side bar charts of feature importance for RF and XGBoost."""
    tree_models = {k: v for k, v in metrics_dict.items()
                   if v['feature_importances'] is not None}
    if not tree_models:
        print('No feature importances available.')
        return

    fig, axes = plt.subplots(1, len(tree_models), figsize=(6 * len(tree_models), 3.5))
    if len(tree_models) == 1:
        axes = [axes]

    for ax, (mname, m) in zip(axes, tree_models.items()):
        imps = m['feature_importances']
        pairs = sorted(zip(valid_cols, imps), key=lambda x: x[1], reverse=True)[:top_n]
        feats, vals = zip(*pairs)
        ax.barh(list(feats)[::-1], list(vals)[::-1], color='steelblue', edgecolor='white')
        ax.set_title(f'{stage_name} — {mname} (top {top_n})', fontsize=10, fontweight='bold')
        ax.set_xlabel('Importance')
        for i, (feat, val) in enumerate(zip(list(feats)[::-1], list(vals)[::-1])):
            ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f'\nTop-{top_n} features ({stage_name}):')
    for mname, m in tree_models.items():
        imps = m['feature_importances']
        pairs = sorted(zip(valid_cols, imps), key=lambda x: x[1], reverse=True)[:top_n]
        print(f'  {mname}:')
        for feat, val in pairs:
            print(f'    {feat:<35} {val:.4f}')


print('Training functions defined.')

## Stage 1 — Script

**Features used:** readability, lexical diversity, prompt coverage, sentence redundancy, entity consistency, topic coherence, factual conflict flag, prompt ambiguity + cross-stage features.

The script stage is the first gate — rejects here save compute on all downstream stages.

In [ ]:
script_best, script_best_name, script_metrics, script_valid_cols, script_X_test, script_y_test = \
    train_stage_classifiers('script', script_features, df)

# Confusion matrices for all three models
fig, axes = plt.subplots(1, len(script_metrics), figsize=(4.5 * len(script_metrics), 3.5))
if len(script_metrics) == 1:
    axes = [axes]
for ax, (mname, m) in zip(axes, script_metrics.items()):
    plot_confusion_matrix(m['cm'], 'Script', mname, ax)
plt.suptitle('Script stage — Confusion matrices', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Feature importance
plot_top_importances(script_metrics, script_valid_cols, 'Script')

## Stage 2 — Audio

**Features used:** phoneme error rate, silence ratio, speaking rate variance, energy variance, TTS word count match + cross-stage features.

`silence_ratio` typically dominates audio quality — long silences indicate TTS failure.

In [ ]:
audio_best, audio_best_name, audio_metrics, audio_valid_cols, audio_X_test, audio_y_test = \
    train_stage_classifiers('audio', audio_features, df)

fig, axes = plt.subplots(1, len(audio_metrics), figsize=(4.5 * len(audio_metrics), 3.5))
if len(audio_metrics) == 1:
    axes = [axes]
for ax, (mname, m) in zip(axes, audio_metrics.items()):
    plot_confusion_matrix(m['cm'], 'Audio', mname, ax)
plt.suptitle('Audio stage — Confusion matrices', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

plot_top_importances(audio_metrics, audio_valid_cols, 'Audio')

## Stage 3 — Visual

**Features used:** CLIP similarity, aesthetic score, blur score, object match score, colour tone match + cross-stage features.

`blur_score` and `aesthetic_score` are typically the strongest predictors of visual rejection.

In [ ]:
visual_best, visual_best_name, visual_metrics, visual_valid_cols, visual_X_test, visual_y_test = \
    train_stage_classifiers('visual', visual_features, df)

fig, axes = plt.subplots(1, len(visual_metrics), figsize=(4.5 * len(visual_metrics), 3.5))
if len(visual_metrics) == 1:
    axes = [axes]
for ax, (mname, m) in zip(axes, visual_metrics.items()):
    plot_confusion_matrix(m['cm'], 'Visual', mname, ax)
plt.suptitle('Visual stage — Confusion matrices', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

plot_top_importances(visual_metrics, visual_valid_cols, 'Visual')

## Stage 4 — Video

**Features used:** AV sync error, transition smoothness, duration deviation + cross-stage features.

`transition_smoothness` dominates video quality. `prior_stage_corrections` carries upstream signal from earlier stages.

In [ ]:
video_best, video_best_name, video_metrics, video_valid_cols, video_X_test, video_y_test = \
    train_stage_classifiers('video', video_features, df)

fig, axes = plt.subplots(1, len(video_metrics), figsize=(4.5 * len(video_metrics), 3.5))
if len(video_metrics) == 1:
    axes = [axes]
for ax, (mname, m) in zip(axes, video_metrics.items()):
    plot_confusion_matrix(m['cm'], 'Video', mname, ax)
plt.suptitle('Video stage — Confusion matrices', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

plot_top_importances(video_metrics, video_valid_cols, 'Video')

## Results Summary

Comparison of all classifiers across all stages. The **best model per stage** (by F1) is highlighted.

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────
all_stage_results = {
    'script': (script_best_name, script_metrics),
    'audio':  (audio_best_name,  audio_metrics),
    'visual': (visual_best_name, visual_metrics),
    'video':  (video_best_name,  video_metrics),
}

rows = []
for stage, (best_name, metrics) in all_stage_results.items():
    for mname, m in metrics.items():
        rows.append({
            'Stage': stage,
            'Model': mname,
            'Accuracy':  m['accuracy'],
            'Precision': m['precision'],
            'Recall':    m['recall'],
            'F1':        m['f1'],
            'ROC_AUC':   m['roc_auc'],
            'Best':      mname == best_name,
        })

summary_df = pd.DataFrame(rows)

def highlight_best(row):
    return ['background-color: #d4edda; font-weight: bold' if row['Best'] else '' for _ in row]

display(
    summary_df.drop(columns='Best')
    .style.apply(highlight_best, axis=1, subset=summary_df.columns.tolist())
    .format({'Accuracy': '{:.3f}', 'Precision': '{:.3f}',
             'Recall': '{:.3f}', 'F1': '{:.3f}', 'ROC_AUC': '{:.3f}'})
    .set_caption('All classifiers — green rows are the best per stage')
)

# ── F1 bar chart ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
pivot = summary_df.pivot(index='Stage', columns='Model', values='F1')
pivot = pivot.reindex(['script', 'audio', 'visual', 'video'])
pivot.plot(kind='bar', ax=ax, edgecolor='white', width=0.65)
ax.set_title('F1 Score comparison — all stages and classifiers', fontsize=12, fontweight='bold')
ax.set_xlabel('Stage')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.0)
ax.tick_params(axis='x', rotation=0)
ax.axhline(0.65, color='red', linestyle='--', linewidth=1, label='Threshold (0.65)')
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')
for bar in ax.patches:
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('\nBest model per stage:')
for stage, (best_name, metrics) in all_stage_results.items():
    f1 = metrics[best_name]['f1']
    auc = metrics[best_name]['roc_auc']
    print(f'  {stage:<8}  {best_name:<22}  F1={f1:.4f}  ROC-AUC={auc:.4f}')

## Save Models

Saved models are loaded at runtime by `backend/ml/predict.py` (Mode C).  
`feature_columns.json` records exactly which columns each model expects, in the correct order.

In [ ]:
import os

# Resolve output directory relative to this notebook's location
NOTEBOOK_DIR = Path(os.path.abspath('__file__') if '__file__' in dir() else os.getcwd())
# Adjust if needed — models always go here:
MODELS_DIR = Path(r'D:\dessertation\shorts_automation\backend\ml\models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

save_map = {
    'script': (script_best,  script_valid_cols),
    'audio':  (audio_best,   audio_valid_cols),
    'visual': (visual_best,  visual_valid_cols),
    'video':  (video_best,   video_valid_cols),
}

feature_columns = {}

for stage, (model, cols) in save_map.items():
    path = MODELS_DIR / f'{stage}_classifier.pkl'
    joblib.dump(model, str(path))
    feature_columns[stage] = cols
    print(f'  Saved {stage}_classifier.pkl  ({path.stat().st_size:,} bytes)')

fc_path = MODELS_DIR / 'feature_columns.json'
fc_path.write_text(json.dumps(feature_columns, indent=2), encoding='utf-8')
print(f'  Saved feature_columns.json  ({fc_path.stat().st_size} bytes)')

print(f'\nAll models saved to: {MODELS_DIR}')
print('\nfeature_columns.json contents:')
print(json.dumps(feature_columns, indent=2))

## Next Steps — Mode C Integration

The four saved `.pkl` files are loaded by **Mode C** (`ml_predictor.py`) at the start of each pipeline run.

### How Mode C uses these models

After each stage completes, the pipeline:

1. Extracts the same quality features used during training
2. Loads the corresponding classifier (e.g. `script_classifier.pkl`)
3. Calls `model.predict_proba(features)[:, 1]` to get **rejection probability** (0–1)
4. Compares the probability to `ML_REJECTION_THRESHOLD = 0.65` (set in `.env`)
5. **If probability ≥ 0.65:** triggers human review (same as Mode B)
6. **If probability < 0.65:** auto-accepts and proceeds to the next stage

### Retraining

Run this notebook again after collecting more Mode B data to improve model accuracy.  
More data — especially more reject examples — will improve recall and F1 on all stages.

| File | Purpose |
|------|---------|
| `backend/ml/models/script_classifier.pkl` | Loaded per script stage attempt |
| `backend/ml/models/audio_classifier.pkl`  | Loaded per audio stage attempt |
| `backend/ml/models/visual_classifier.pkl` | Loaded per visual stage attempt |
| `backend/ml/models/video_classifier.pkl`  | Loaded per video stage attempt |
| `backend/ml/models/feature_columns.json`  | Column order for each model's input |
| `backend/ml/models/training_results.json` | Metrics from last training run |